# Tutorial: py-SpotSweeper

SpotSweeper provides spatially-aware quality control for spot-based spatial transcriptomics (e.g., 10x Visium). It identifies outlier spots and technical artifacts using kNN-based local statistics.

**Reference**: Totty et al. (2025) "SpotSweeper: spatially-aware quality control for spatial transcriptomics." Bioconductor.

In [ ]:
# pip install spotsweeper
import numpy as np
import pandas as pd
from scipy.io import mmread
from scipy.sparse import csr_matrix
import anndata as ad
import matplotlib.pyplot as plt

from spotsweeper import local_variance, local_outliers, find_artifacts, flag_visium_outliers

## Load demo data

In [ ]:
DATA_DIR = '../data'
counts = mmread(f'{DATA_DIR}/fixture_counts.mtx').T
coords = pd.read_csv(f'{DATA_DIR}/fixture_spatial_coords.csv', index_col=0)
metadata = pd.read_csv(f'{DATA_DIR}/fixture_metadata.csv', index_col=0)
features = pd.read_csv(f'{DATA_DIR}/feature_names.csv')
spots = pd.read_csv(f'{DATA_DIR}/spot_names.csv')
adata = ad.AnnData(X=csr_matrix(counts), obs=metadata,
                   var=pd.DataFrame(index=features.iloc[:,0].values))
adata.obs_names = spots.iloc[:,0].values
adata.obsm['spatial'] = coords.values
print(f'Dataset: {adata.shape[0]} spots x {adata.shape[1]} genes')
print(f'Columns: {list(adata.obs.columns)}')

## 1. localVariance — Local variance of QC metrics

Computes the variance of a metric within kNN neighborhoods, then regresses out mean-variance bias using robust linear regression. The residuals represent local variance adjusted for the mean-variance relationship.

```r
# R equivalent
spe <- localVariance(spe, metric='subsets_Mito_percent', n_neighbors=36,
                     name='local_mito_variance_k36', workers=1)
```

In [ ]:
adata = local_variance(adata, metric='subsets_Mito_percent', n_neighbors=36,
                       name='local_mito_variance_k36', log=False)
print(f'Output: adata.obs["local_mito_variance_k36"]')
print(f'Shape: {adata.obs["local_mito_variance_k36"].shape}')
print(f'Sample: {adata.obs["local_mito_variance_k36"].values[:5]}')

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(adata.obsm['spatial'][:,0], -adata.obsm['spatial'][:,1],
                c=adata.obs['local_mito_variance_k36'], s=1, cmap='RdBu_r')
plt.colorbar(sc, label='Local mito variance')
ax.set_title('Local Mitochondrial Variance'); ax.set_aspect('equal')
plt.show()

## 2. localOutliers — Detect local outliers

Detects outliers using modified z-scores (MAD-based) within kNN neighborhoods. Spots with z-scores exceeding the cutoff in the specified direction are flagged.

```r
# R equivalent
spe <- localOutliers(spe, metric='sum', direction='lower', log=TRUE)
```

In [ ]:
adata = local_outliers(adata, metric='sum', direction='lower', log=True)
print(f'Outliers: {adata.obs["sum_outliers"].sum()} / {len(adata)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(adata.obsm['spatial'][:,0], -adata.obsm['spatial'][:,1],
                     c=adata.obs['sum_z'], s=1, cmap='RdBu_r')
plt.colorbar(sc, ax=axes[0], label='Z-score')
axes[0].set_title('Local Outlier Z-scores'); axes[0].set_aspect('equal')

outlier_mask = adata.obs['sum_outliers'].values.astype(bool)
axes[1].scatter(adata.obsm[~outlier_mask,0], -adata.obsm[~outlier_mask,1],
                c='gray', s=1, label='Normal')
axes[1].scatter(adata.obsm[outlier_mask,0], -adata.obsm[outlier_mask,1],
                c='red', s=10, label='Outlier')
axes[1].set_title('Outlier Spots'); axes[1].set_aspect('equal'); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. flagVisiumOutliers — Flag systematic Visium outliers

Flags known systematic technical outlier spots in Visium arrays based on barcode matching.

```r
# R equivalent
spe <- flagVisiumOutliers(spe)
```

In [ ]:
adata = flag_visium_outliers(adata)
print(f'Systematic outliers: {adata.obs["systematic_outliers"].sum()} / {len(adata)}')

fig, ax = plt.subplots(figsize=(8, 6))
sys_out = adata.obs['systematic_outliers'].values.astype(bool)
ax.scatter(adata.obsm['spatial'][~sys_out,0], -adata.obsm['spatial'][~sys_out,1],
           c='gray', s=1, label='Normal')
ax.scatter(adata.obsm['spatial'][sys_out,0], -adata.obsm['spatial'][sys_out,1],
           c='red', s=20, label='Systematic outlier')
ax.set_title('Systematic Visium Outliers'); ax.set_aspect('equal'); ax.legend()
plt.show()

## Common pitfalls / FAQ

1. **Coordinate system**: `adata.obsm['spatial']` must be an (n_spots, 2) array with x/y coordinates.
2. **Sample IDs**: Functions iterate per-sample. Ensure `adata.obs[samples]` is set correctly.
3. **kNN tie-breaking**: Different kNN implementations may break ties differently. For exact R parity, pass R-exported kNN indices via `knn_indices` parameter.
4. **Log transform**: `localVariance` and `localOutliers` default to `log=True`/`log=False` differently — check defaults.

## Where to go next

- [README.md](../README.md)
- [RECONSTRUCTION_REPORT.md](../RECONSTRUCTION_REPORT.md)
- [R SpotSweeper](https://github.com/MicTott/SpotSweeper)